In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import json
from pathlib import Path


class HyenadnaEmbeddingExtractor:
    def __init__(self,
                 checkpoint: str = 'LongSafari/hyenadna-medium-160k-seqlen-hf',
                 device: str = None,
                 max_length: int = 160_000):
        """
        Класс для извлечения эмбеддингов из последовательностей ДНК/РНК с помощью модели Hyenadna.
        :param checkpoint: Название модели на Hugging Face
        :param device: 'cpu' или 'cuda'. По умолчанию определяется автоматически.
        :param max_length: Максимальная длина токенизированной последовательности.
        """
        self.checkpoint = checkpoint
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.max_length = max_length
        
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint, trust_remote_code=True)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            output_hidden_states=True
        )
        self.model.eval()

    def extract_embeddings(self, sequences, batch_size: int = 8):
        """
        Извлекает эмбеддинги для списка последовательностей.
        :param sequences: Список строк ДНК/РНК
        :param batch_size: Размер батча для инференса
        :return: numpy.ndarray размера (len(sequences), hidden_size)
        """
        all_embeddings = []
        for i in tqdm(range(0, len(sequences), batch_size), desc="Extract embeddings"):
            batch_seqs = sequences[i:i + batch_size]
            inputs = self.tokenizer(
                batch_seqs,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors='pt'
            )
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = self.model(**inputs)
                # Берём последнее скрытое состояние
                hidden_states = outputs.hidden_states[-1]  # (batch, seq_len, hidden_size)
                # Среднее по длине последовательности
                embs = hidden_states.mean(dim=1).to(torch.float32)         # (batch, hidden_size)
                embs = embs.cpu().numpy()
            all_embeddings.append(embs)

        return np.vstack(all_embeddings)


PARAMS_LOGREG = {'max_iter': 1000, 'random_state': 42}
PATH_TO_SAVE_OUTPUTS = '/kaggle/working'
BATCH_SIZE = 16

ds = load_dataset(
    "InstaDeepAI/nucleotide_transformer_downstream_tasks",
    split=None,
    trust_remote_code=True
)
train_ds, test_ds = ds['train'], ds['test']

extractor = HyenadnaEmbeddingExtractor()

# Baseline: обучение на полном наборе
baseline = {}
for task in tqdm(set(train_ds['task']), desc='Baseline'):
    tr = train_ds.filter(lambda x, t=task: x['task'] == t)
    te = test_ds.filter(lambda x, t=task: x['task'] == t)
    seqs_tr = tr['sequence']
    y_tr = np.array(tr['label'])
    seqs_te = te['sequence']
    y_te = np.array(te['label'])

    X_tr = extractor.extract_embeddings(seqs_tr, batch_size=BATCH_SIZE)
    X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)

    clf = LogisticRegression(**PARAMS_LOGREG)
    # Поддержка одномерных эмбеддингов
    Xf = X_tr.reshape(-1, 1) if X_tr.ndim == 1 or X_tr.shape[1] == 1 else X_tr
    Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
    clf.fit(Xf, y_tr)
    preds = clf.predict(Xt)

    baseline[task] = {
        'accuracy': float(accuracy_score(y_te, preds)),
        'f1_score': float(f1_score(y_te, preds, average='macro'))
    }
    with open(f'{PATH_TO_SAVE_OUTPUTS}/results_hyenadna_task-{task}_baseline.json', 'w') as f:
        json.dump(baseline, f, indent=4)

# Few-shot эксперименты
def few_shot(train, test, ks=(1, 5, 10, 20), trials=5):
    results = {}
    rng = np.random.RandomState(42)
    for task in tqdm(set(train['task']), desc='Few-shot'):
        tr = train.filter(lambda x, t=task: x['task'] == t)
        te = test.filter(lambda x, t=task: x['task'] == t)
        seqs_tr = tr['sequence']
        y_tr = np.array(tr['label'])
        seqs_te = te['sequence']
        y_te = np.array(te['label'])

        # Предвычисление эмбеддингов для теста
        X_te = extractor.extract_embeddings(seqs_te, batch_size=BATCH_SIZE)
        results[task] = {}

        for k in ks:
            accs, f1s = [], []
            for _ in range(trials):
                idxs = []
                for lbl in np.unique(y_tr):
                    locs = np.where(y_tr == lbl)[0]
                    choice = rng.choice(locs, size=min(k, len(locs)), replace=False)
                    idxs.extend(choice.tolist())
                # Извлекаем k-shot эмбеддинги
                X_k = extractor.extract_embeddings([seqs_tr[i] for i in idxs], batch_size=BATCH_SIZE)
                y_k = y_tr[idxs]

                clf = LogisticRegression(**PARAMS_LOGREG)
                Xf_k = X_k.reshape(-1, 1) if X_k.ndim == 1 or X_k.shape[1] == 1 else X_k
                Xt = X_te.reshape(-1, 1) if X_te.ndim == 1 or X_te.shape[1] == 1 else X_te
                clf.fit(Xf_k, y_k)
                preds = clf.predict(Xt)
                accs.append(accuracy_score(y_te, preds))
                f1s.append(f1_score(y_te, preds, average='macro'))

            results[task][k] = {
                'accuracy': float(np.mean(accs)),
                'f1_score': float(np.mean(f1s))
            }
            with open(f'{PATH_TO_SAVE_OUTPUTS}/results_hyenadna_task-{task}_k-{k}.json', 'w') as f:
                json.dump(results, f, indent=4)
    return results

results_kshot = few_shot(train_ds, test_ds)

output = {'full': baseline, 'kshot': results_kshot, 'params': PARAMS_LOGREG}
with open(f'{PATH_TO_SAVE_OUTPUTS}/results_hyenadna.json', 'w') as f:
    json.dump(output, f, indent=4)


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Some weights of HyenaDNAForSequenceClassification were not initialized from the model checkpoint at LongSafari/hyenadna-medium-160k-seqlen-hf and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Baseline:   0%|          | 0/18 [00:00<?, ?it/s]

Extract embeddings:   0%|          | 0/2070 [00:00<?, ?it/s]

...

Extract embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Extract embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Extract embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Extract embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Filter:   0%|          | 0/461850 [00:00<?, ? examples/s]